# Dual-Agent Ensemble Evaluation — M1 + M4 (P1 Q2 INASS Paper)

Notebook ini mengevaluasi **5 strategi agregasi** untuk dual-agent ensemble:
- **M1 (NV-Reason-CXR-3B)** sebagai **Detector** — high sensitivity, over-predicts, reasoning-style output
- **M4 (MedGemma-4B-IT)** sebagai **Reviewer** — balanced response, gradual ICL, medical-general training

### Motivasi M1+M4 vs M1+M2
Ensemble M1+M2 sebelumnya **gagal** karena M2 adalah "dead agent" (sensitivity 2-17%). M4 memiliki:
- Sensitivity lebih tinggi (0.142→0.454 dari k=0→k=8) vs M2 (0.170→~0.20)
- Respons ICL gradual & proporsional (bukan resisten seperti M2)
- Arsitektur berbeda (Gemma 3 vs Qwen2.5-VL) — diversitas arsitektur
- Generalization gap yang menghilang pada k=8

### 5 Strategi Ensemble
| ID | Strategi | Deskripsi |
|---|---|---|
| S1 | Unanimous Positive | PTX hanya jika **kedua** model setuju |
| S2 | Any Positive | PTX jika **salah satu** model mendeteksi |
| S3 | Detector→Reviewer | M1 screens; jika positif, M4 memverifikasi |
| S4 | PBR-Weighted | Weight berdasarkan kedekatan PBR ke 1.0 |
| S5 | Confidence-Weighted | Weight berdasarkan confidence ekstraksi jawaban |

## 1 — Setup & Imports

In [ ]:
import sys
import json
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path('src').resolve()))
from ensemble import (
    load_paired, run_ensemble, run_single_model_baselines,
    compare_all, compute_pbr, compute_sdr, compute_pbr_table,
    STRATEGIES, STRATEGY_FNS,
)
from evaluate import extract_diagnosis, compute_classification_metrics

# Style
matplotlib.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_palette('colorblind')

# Paths — M1 + M4
M1_PATH = Path('result-M1/all_results.jsonl')
M4_PATH = Path('result-M4/all_results.jsonl')  # M4 dimuat sebagai 'M2' secara internal
FIGURES_DIR = Path('figures-ensemble-m1-m4')
RESULTS_DIR = Path('results-ensemble-m1-m4')
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 160)

# Label mapping: internal column names → display labels
# ensemble.py uses pred_m1/pred_m2 internally, but we display as M1/M4
LABEL_M1 = 'M1 (NV-Reason-CXR-3B)'
LABEL_M4 = 'M4 (MedGemma-4B-IT)'
LABEL_SHORT_M1 = 'M1'
LABEL_SHORT_M4 = 'M4'

print('Setup complete.')
print(f'M1 path: {M1_PATH} — exists: {M1_PATH.exists()}')
print(f'M4 path: {M4_PATH} — exists: {M4_PATH.exists()}')

## 2 — Load & Pair Data

In [ ]:
# load_paired() internally uses pred_m1/pred_m2 columns
# M4's predictions will be in pred_m2 column
df_paired = load_paired(M1_PATH, M4_PATH)

print(f'\nShape: {df_paired.shape}')
print(f'Columns: {list(df_paired.columns)}')
print(f'k_shots: {sorted(df_paired["k_shot"].unique())}')
print(f'seeds: {sorted(df_paired["seed"].unique())}')
print(f'test_sets: {sorted(df_paired["test_set"].unique())}')

print(f'\nNote: pred_m2/conf_m2 columns contain M4 data')
print(f'  pred_m1 (M1) distribution: {df_paired["pred_m1"].value_counts().to_dict()}')
print(f'  pred_m2 (M4) distribution: {df_paired["pred_m2"].value_counts().to_dict()}')

## 3 — Quick Sanity Check: Agreement M1 vs M4

In [ ]:
# Agreement per k-shot
print('Agreement Rate M1 vs M4 per (k_shot, test_set):')
print('=' * 70)
for (k, ts), grp in df_paired.groupby(['k_shot', 'test_set']):
    agree = (grp['pred_m1'] == grp['pred_m2']).mean()
    both_pos = ((grp['pred_m1'] == 1) & (grp['pred_m2'] == 1)).mean()
    both_neg = ((grp['pred_m1'] == 0) & (grp['pred_m2'] == 0)).mean()
    m1_only = ((grp['pred_m1'] == 1) & (grp['pred_m2'] == 0)).mean()
    m4_only = ((grp['pred_m1'] == 0) & (grp['pred_m2'] == 1)).mean()
    short_ts = 'Int' if 'Internal' in ts else 'Ext'
    print(f'  k={k} {short_ts:3s}  |  Agree={agree:.3f}  '
          f'Both+={both_pos:.3f}  Both-={both_neg:.3f}  '
          f'M1only+={m1_only:.3f}  M4only+={m4_only:.3f}')

## 4 — PBR Table (M1 & M4 Individual)

In [ ]:
pbr_table = compute_pbr_table(df_paired)
print('PBR per (k_shot, test_set) — pooled over seeds:')
print('  PBR=1.0 is ideal. >1 = over-predict, <1 = under-predict')
print('  Note: pbr_m2 column = M4 (MedGemma)')
print('=' * 60)
for _, row in pbr_table.iterrows():
    short_ts = 'Int' if 'Internal' in row['test_set'] else 'Ext'
    print(f'  k={int(row["k_shot"]):1d} {short_ts:3s}  |  '
          f'PBR_M1={row["pbr_m1"]:.3f}  PBR_M4={row["pbr_m2"]:.3f}')

## 5 — Run Ensemble Evaluation (All Strategies + Baselines)

In [ ]:
# Pooled over seeds (for main results)
df_results = compare_all(df_paired, per_seed=False)
df_results.to_csv(RESULTS_DIR / 'ensemble_results_pooled.csv', index=False)

print(f'Total rows: {len(df_results)}')
print(f'Strategies: {df_results["strategy"].unique().tolist()}')
print('\nNote: M1_single = M1, M2_single = M4 (MedGemma)')
df_results.head(20)

In [ ]:
# Per-seed (for variance analysis & statistical tests)
df_per_seed = compare_all(df_paired, per_seed=True)
df_per_seed.to_csv(RESULTS_DIR / 'ensemble_results_per_seed.csv', index=False)

print(f'Per-seed rows: {len(df_per_seed)}')

## 6 — Summary Table: Mean ± Std per Strategy

In [ ]:
METRIC_COLS = ['sensitivity', 'specificity', 'precision', 'f1', 'accuracy', 'pbr', 'sdr']

# Compute mean ± std across seeds
summary_rows = []
for (strat, k, ts), grp in df_per_seed.groupby(['strategy', 'k_shot', 'test_set']):
    row = {'strategy': strat, 'k_shot': k, 'test_set': ts}
    for col in METRIC_COLS:
        vals = grp[col].dropna()
        if len(vals) > 0:
            row[f'{col}_mean'] = vals.mean()
            row[f'{col}_std'] = vals.std()
        else:
            row[f'{col}_mean'] = None
            row[f'{col}_std'] = None
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / 'ensemble_summary_mean_std.csv', index=False)

# Display formatted
print('Summary Table (Mean ± Std over 5 seeds):')
print('Note: M2_single = M4 (MedGemma-4B-IT)')
print('=' * 120)

for ts in sorted(summary_df['test_set'].unique()):
    short_ts = 'CheXpert (Internal)' if 'Internal' in ts else 'RS UNAIR (External)'
    print(f'\n--- {short_ts} ---')
    sub = summary_df[summary_df['test_set'] == ts].sort_values(['k_shot', 'strategy'])
    for _, r in sub.iterrows():
        strat_label = r['strategy'].replace('M2_single', 'M4_single')
        print(f"  k={int(r['k_shot'])}  {strat_label:20s}  "
              f"F1={r['f1_mean']:.3f}±{r['f1_std']:.3f}  "
              f"Sens={r['sensitivity_mean']:.3f}±{r['sensitivity_std']:.3f}  "
              f"Spec={r['specificity_mean']:.3f}±{r['specificity_std']:.3f}  "
              f"PBR={r['pbr_mean']:.3f}")

## 7 — Figure 1: F1 Score Comparison (Single vs Ensemble per k-shot)

In [ ]:
STRATEGY_LABELS = {
    'M1_single': 'M1 (Detector)',
    'M2_single': 'M4 (Reviewer)',
    'S1_unanimous': 'S1: Unanimous',
    'S2_any': 'S2: Any Positive',
    'S3_detect_verify': 'S3: Detect→Review',
    'S4_pbr_weighted': 'S4: PBR-Weighted',
    'S5_conf_weighted': 'S5: Conf-Weighted',
}

STRATEGY_STYLES = {
    'M1_single': {'ls': '--', 'marker': 's', 'lw': 2},
    'M2_single': {'ls': '--', 'marker': 'D', 'lw': 2},
    'S1_unanimous': {'ls': '-', 'marker': 'o', 'lw': 1.5},
    'S2_any': {'ls': '-', 'marker': '^', 'lw': 1.5},
    'S3_detect_verify': {'ls': '-', 'marker': 'v', 'lw': 1.5},
    'S4_pbr_weighted': {'ls': '-', 'marker': 'P', 'lw': 1.5},
    'S5_conf_weighted': {'ls': '-', 'marker': 'X', 'lw': 1.5},
}

all_strats = ['M1_single', 'M2_single'] + STRATEGIES
test_sets = sorted(summary_df['test_set'].unique())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in all_strats:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        style = STRATEGY_STYLES[strat]
        ax.errorbar(s['k_shot'], s['f1_mean'], yerr=s['f1_std'],
                    label=STRATEGY_LABELS[strat], **style,
                    capsize=3, markersize=6)
    
    short_ts = 'CheXpert (Internal)' if 'Internal' in ts else 'RS UNAIR (External)'
    ax.set_title(f'F1 Score — {short_ts}', fontsize=13, fontweight='bold')
    ax.set_xlabel('k per class')
    ax.set_ylabel('F1 Score')
    ax.set_ylim(-0.05, 1.0)
    ax.legend(fontsize=8, loc='upper left')

fig.suptitle('M1+M4 Ensemble: F1 Score Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_f1_comparison.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig1_f1_comparison.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig1_f1_comparison')

## 8 — Figure 2: Sensitivity & Specificity per Strategy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single', 'S2_any', 'S3_detect_verify', 'S4_pbr_weighted']:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        style = STRATEGY_STYLES[strat]
        label = STRATEGY_LABELS[strat]
        ax.plot(s['k_shot'], s['sensitivity_mean'],
                label=f'{label} (Sens)', **style, markersize=5)
        ax.plot(s['k_shot'], s['specificity_mean'],
                label=f'{label} (Spec)', **{**style, 'ls': ':'}, markersize=5, alpha=0.6)
    
    short_ts = 'CheXpert (Internal)' if 'Internal' in ts else 'RS UNAIR (External)'
    ax.set_title(f'Sensitivity & Specificity — {short_ts}', fontsize=13, fontweight='bold')
    ax.set_xlabel('k per class')
    ax.set_ylabel('Score')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=6, loc='center left', ncol=1)

fig.suptitle('M1+M4 Ensemble: Sensitivity & Specificity Trade-off', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_sens_spec.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig2_sens_spec.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig2_sens_spec')

## 9 — Figure 3: PBR Trajectory (Closer to 1.0 = Better)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single', 'S1_unanimous', 'S2_any', 'S3_detect_verify', 'S4_pbr_weighted']:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        style = STRATEGY_STYLES[strat]
        ax.errorbar(s['k_shot'], s['pbr_mean'], yerr=s['pbr_std'],
                    label=STRATEGY_LABELS[strat], **style,
                    capsize=3, markersize=6)
    
    # Ideal PBR line
    ax.axhline(y=1.0, color='red', ls='--', alpha=0.5, label='Ideal PBR=1.0')
    
    short_ts = 'CheXpert (Internal)' if 'Internal' in ts else 'RS UNAIR (External)'
    ax.set_title(f'PBR Trajectory — {short_ts}', fontsize=13, fontweight='bold')
    ax.set_xlabel('k per class')
    ax.set_ylabel('PBR (Prediction Bias Ratio)')
    ax.legend(fontsize=8, loc='best')

fig.suptitle('M1+M4 Ensemble: Prediction Bias Ratio', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_pbr_trajectory.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig3_pbr_trajectory.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig3_pbr_trajectory')

## 10 — Figure 4: SDR Comparison (Lower = Better)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single', 'S1_unanimous', 'S2_any', 'S3_detect_verify', 'S4_pbr_weighted']:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        style = STRATEGY_STYLES[strat]
        ax.errorbar(s['k_shot'], s['sdr_mean'], yerr=s['sdr_std'],
                    label=STRATEGY_LABELS[strat], **style,
                    capsize=3, markersize=6)
    
    ax.axhline(y=0, color='gray', ls='--', alpha=0.5)
    
    short_ts = 'CheXpert (Internal)' if 'Internal' in ts else 'RS UNAIR (External)'
    ax.set_title(f'SDR — {short_ts}', fontsize=13, fontweight='bold')
    ax.set_xlabel('k per class')
    ax.set_ylabel('SDR (%) — Specificity Drop Rate')
    ax.legend(fontsize=8, loc='best')

fig.suptitle('M1+M4 Ensemble: Specificity Drop Rate', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_sdr.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig4_sdr.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig4_sdr')

## 11 — Figure 5: Bar Chart — Best k per Strategy (F1)

In [ ]:
# Find best k per strategy per test set
best_rows = []
for (strat, ts), grp in summary_df.groupby(['strategy', 'test_set']):
    best = grp.loc[grp['f1_mean'].idxmax()]
    best_rows.append({
        'strategy': strat,
        'test_set': ts,
        'best_k': int(best['k_shot']),
        'f1_mean': best['f1_mean'],
        'f1_std': best['f1_std'],
        'sensitivity_mean': best['sensitivity_mean'],
        'specificity_mean': best['specificity_mean'],
        'pbr_mean': best['pbr_mean'],
    })

best_df = pd.DataFrame(best_rows)
best_df.to_csv(RESULTS_DIR / 'best_k_per_strategy.csv', index=False)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = best_df[best_df['test_set'] == ts].sort_values('f1_mean', ascending=True)
    
    labels = [STRATEGY_LABELS.get(s, s) for s in sub['strategy']]
    colors = ['#2196F3' if 'single' in s else '#FF9800' for s in sub['strategy']]
    
    bars = ax.barh(labels, sub['f1_mean'], xerr=sub['f1_std'],
                   color=colors, capsize=3, edgecolor='black', linewidth=0.5)
    
    # Annotate with best_k and F1 value
    for bar, (_, r) in zip(bars, sub.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'k={int(r["best_k"])} | F1={r["f1_mean"]:.3f}',
                va='center', fontsize=9)
    
    short_ts = 'CheXpert (Internal)' if 'Internal' in ts else 'RS UNAIR (External)'
    ax.set_title(f'Best F1 per Strategy — {short_ts}', fontsize=13, fontweight='bold')
    ax.set_xlabel('F1 Score')
    ax.set_xlim(0, 1.0)

fig.suptitle('M1+M4 Ensemble: Best F1 at Optimal k', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_best_k_bar.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig5_best_k_bar.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig5_best_k_bar')

# Print best-k table
print('\nBest k per strategy:')
for _, r in best_df.sort_values(['test_set', 'f1_mean'], ascending=[True, False]).iterrows():
    strat_label = STRATEGY_LABELS.get(r['strategy'], r['strategy'])
    short_ts = 'Int' if 'Internal' in r['test_set'] else 'Ext'
    print(f'  {short_ts:3s}  {strat_label:20s}  best_k={int(r["best_k"])}  '
          f'F1={r["f1_mean"]:.3f}  Sens={r["sensitivity_mean"]:.3f}  Spec={r["specificity_mean"]:.3f}')

## 12 — Figure 6: Scatter Plot — Sensitivity vs Specificity Trade-off

In [ ]:
# One point per (strategy, k) — shows the sens-spec trade-off space
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in all_strats:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        style = STRATEGY_STYLES[strat]
        ax.scatter(s['specificity_mean'], s['sensitivity_mean'],
                   label=STRATEGY_LABELS[strat],
                   marker=style['marker'], s=80, zorder=3)
        # Connect points with lines
        ax.plot(s['specificity_mean'], s['sensitivity_mean'],
                ls=style['ls'], lw=style['lw'], alpha=0.5)
        # Annotate k values
        for _, r in s.iterrows():
            ax.annotate(f'k={int(r["k_shot"])}',
                        (r['specificity_mean'], r['sensitivity_mean']),
                        fontsize=6, alpha=0.7,
                        xytext=(3, 3), textcoords='offset points')
    
    # Ideal corner
    ax.plot(1.0, 1.0, '*', color='red', markersize=15, zorder=5, label='Ideal')
    
    short_ts = 'CheXpert (Internal)' if 'Internal' in ts else 'RS UNAIR (External)'
    ax.set_title(f'Sens-Spec Trade-off — {short_ts}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Specificity')
    ax.set_ylabel('Sensitivity')
    ax.set_xlim(-0.05, 1.1)
    ax.set_ylim(-0.05, 1.1)
    ax.legend(fontsize=7, loc='lower left')

fig.suptitle('M1+M4 Ensemble: Sensitivity vs Specificity Space', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig6_sens_spec_scatter.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig6_sens_spec_scatter.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig6_sens_spec_scatter')

## 13 — Figure 7: Agreement Rate & Conflict Resolution

In [ ]:
# Use pooled results for agreement/conflict analysis
ens_only = df_results[~df_results['strategy'].str.contains('single')].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Agreement rate
ax = axes[0]
for ts in test_sets:
    sub = ens_only[(ens_only['test_set'] == ts) & (ens_only['strategy'] == 'S1_unanimous')]
    sub = sub.sort_values('k_shot')
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    ax.plot(sub['k_shot'], sub['agreement_rate'],
            marker='o', lw=2, label=short_ts)

ax.set_title('M1-M4 Agreement Rate', fontsize=13, fontweight='bold')
ax.set_xlabel('k per class')
ax.set_ylabel('Agreement Rate')
ax.set_ylim(0, 1.05)
ax.legend()

# (b) Conflict resolution accuracy per strategy
ax = axes[1]
for strat in STRATEGIES:
    sub = ens_only[(ens_only['strategy'] == strat) & 
                   (ens_only['test_set'] == test_sets[0])].sort_values('k_shot')
    conflict_acc = sub['conflict_resolution_acc'].dropna()
    if len(conflict_acc) > 0:
        ax.plot(sub['k_shot'], sub['conflict_resolution_acc'],
                label=STRATEGY_LABELS[strat],
                **STRATEGY_STYLES[strat], markersize=6)

ax.axhline(y=0.5, color='gray', ls='--', alpha=0.5, label='Random')
ax.set_title('Conflict Resolution Accuracy (Internal)', fontsize=13, fontweight='bold')
ax.set_xlabel('k per class')
ax.set_ylabel('Accuracy on Disagreement Cases')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)

fig.suptitle('M1+M4 Ensemble: Agreement & Conflict Resolution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_agreement_conflict.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig7_agreement_conflict.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig7_agreement_conflict')

## 14 — Figure 8: Heatmap — All Metrics at Best-k (Internal)

In [ ]:
# Heatmap of metrics at best-k for each strategy (Internal test set)
ts_internal = 'CheXpert Plus (Internal)'
heatmap_metrics = ['f1', 'sensitivity', 'specificity', 'precision', 'accuracy']

best_internal = best_df[best_df['test_set'] == ts_internal].copy()

# Get full metrics at best-k for each strategy
heat_rows = []
for _, br in best_internal.iterrows():
    match = summary_df[
        (summary_df['strategy'] == br['strategy']) &
        (summary_df['test_set'] == ts_internal) &
        (summary_df['k_shot'] == br['best_k'])
    ]
    if len(match) > 0:
        r = match.iloc[0]
        label = STRATEGY_LABELS.get(br['strategy'], br['strategy'])
        row = {'Strategy': f"{label} (k={int(br['best_k'])})"}
        for m in heatmap_metrics:
            row[m.capitalize()] = r[f'{m}_mean']
        heat_rows.append(row)

heat_df = pd.DataFrame(heat_rows).set_index('Strategy')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(heat_df, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('M1+M4 Ensemble: Metrics at Best-k (Internal)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig8_heatmap_bestk.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig8_heatmap_bestk.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig8_heatmap_bestk')

## 15 — Figure 9: Cross-Institutional Robustness (Internal vs External)

In [ ]:
# F1 gap between internal and external for each strategy at best-k
gap_rows = []
for strat in all_strats:
    int_best = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == 'CheXpert Plus (Internal)')]
    ext_best = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == 'RS UNAIR (External)')]
    if len(int_best) > 0 and len(ext_best) > 0:
        gap_rows.append({
            'strategy': STRATEGY_LABELS.get(strat, strat),
            'f1_internal': int_best.iloc[0]['f1_mean'],
            'f1_external': ext_best.iloc[0]['f1_mean'],
            'f1_gap': int_best.iloc[0]['f1_mean'] - ext_best.iloc[0]['f1_mean'],
            'best_k_int': int(int_best.iloc[0]['best_k']),
            'best_k_ext': int(ext_best.iloc[0]['best_k']),
        })

gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(RESULTS_DIR / 'cross_institutional_gap.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Side-by-side F1
ax = axes[0]
x = np.arange(len(gap_df))
w = 0.35
ax.bar(x - w/2, gap_df['f1_internal'], w, label='Internal', color='#2196F3')
ax.bar(x + w/2, gap_df['f1_external'], w, label='External', color='#FF9800')
ax.set_xticks(x)
ax.set_xticklabels(gap_df['strategy'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('F1 Score')
ax.set_title('F1 at Best-k: Internal vs External', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.0)

# (b) Gap
ax = axes[1]
colors = ['green' if g < 0 else 'orange' for g in gap_df['f1_gap']]
ax.barh(gap_df['strategy'], gap_df['f1_gap'], color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(x=0, color='black', lw=0.5)
ax.set_xlabel('F1 Gap (Internal - External)')
ax.set_title('Generalization Gap at Best-k', fontsize=13, fontweight='bold')

for i, (_, r) in enumerate(gap_df.iterrows()):
    ax.text(r['f1_gap'] + 0.005 if r['f1_gap'] >= 0 else r['f1_gap'] - 0.005,
            i, f'{r["f1_gap"]:+.3f}', va='center',
            ha='left' if r['f1_gap'] >= 0 else 'right', fontsize=9)

fig.suptitle('M1+M4 Ensemble: Cross-Institutional Robustness', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig9_cross_institutional.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig9_cross_institutional.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig9_cross_institutional')

# Print table
print('\nCross-Institutional Gap:')
for _, r in gap_df.iterrows():
    print(f'  {r["strategy"]:20s}  Int={r["f1_internal"]:.3f}(k={r["best_k_int"]})  '
          f'Ext={r["f1_external"]:.3f}(k={r["best_k_ext"]})  Gap={r["f1_gap"]:+.3f}')

## 16 — Figure 10: Confusion Matrices — Best Strategy vs Single Models

In [ ]:
# Identify the best ensemble strategy (by F1 on internal)
int_best = best_df[best_df['test_set'] == 'CheXpert Plus (Internal)']
ens_best = int_best[~int_best['strategy'].str.contains('single')]
best_strat = ens_best.loc[ens_best['f1_mean'].idxmax(), 'strategy']
best_k_ens = int(ens_best.loc[ens_best['f1_mean'].idxmax(), 'best_k'])

print(f'Best ensemble strategy: {STRATEGY_LABELS[best_strat]} at k={best_k_ens}')

# Build confusion matrices for: M1_single, M4_single (M2_single), best ensemble
# at the best_k for internal
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for row_idx, ts in enumerate(test_sets):
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    sub = df_paired[(df_paired['k_shot'] == best_k_ens) & (df_paired['test_set'] == ts)]
    gt = sub['ground_truth'].values
    
    configs = [
        ('M1 (Detector)', sub['pred_m1'].values),
        ('M4 (Reviewer)', sub['pred_m2'].values),  # pred_m2 = M4
        (STRATEGY_LABELS[best_strat], STRATEGY_FNS[best_strat](
            sub['pred_m1'].values, sub['pred_m2'].values,
            pbr_m1=compute_pbr(gt, sub['pred_m1'].values),
            pbr_m2=compute_pbr(gt, sub['pred_m2'].values),
            conf_m1=sub['conf_m1'].values,
            conf_m2=sub['conf_m2'].values,
        )),
    ]
    
    for col_idx, (name, preds) in enumerate(configs):
        ax = axes[row_idx, col_idx]
        metrics = compute_classification_metrics(gt, preds)
        
        # Build confusion matrix
        tp = metrics['tp']
        tn = metrics['tn']
        fp = metrics['fp']
        fn = metrics['fn']
        total = tp + tn + fp + fn
        cm = np.array([[tn, fp], [fn, tp]])
        
        # Plot
        sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Normal (0)', 'PTX (1)'],
                    yticklabels=['Normal (0)', 'PTX (1)'])
        
        # Annotate with counts and percentages
        for i in range(2):
            for j in range(2):
                val = cm[i, j]
                pct = val / total * 100
                ax.text(j + 0.5, i + 0.5,
                        f'{val}\n({pct:.1f}%)',
                        ha='center', va='center', fontsize=12,
                        color='white' if val > total*0.3 else 'black')
        
        ax.set_title(f'{short_ts} — {name}\n'
                     f'(Acc={metrics["accuracy"]:.3f}, F1={metrics["f1"]:.3f})',
                     fontsize=11, fontweight='bold')
        ax.set_ylabel('Ground Truth')
        ax.set_xlabel('Predicted')

fig.suptitle(f'M1+M4 Ensemble: Confusion Matrices at k={best_k_ens}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig10_confusion_matrices.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig10_confusion_matrices.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig10_confusion_matrices')

## 17 — Statistical Significance Tests

In [ ]:
# Test: Ensemble strategy vs best single model (Wilcoxon signed-rank per seed)
print('=' * 80)
print('STATISTICAL TESTS: Ensemble vs Single Model (Wilcoxon signed-rank, per seed)')
print('=' * 80)
print('H0: Ensemble F1 = best single model F1')
print('H1: Ensemble F1 ≠ best single model F1')
print('Note: M2_single = M4 (MedGemma-4B-IT)')

ts_int = 'CheXpert Plus (Internal)'
ts_ext = 'RS UNAIR (External)'

for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'\n--- {short_ts} ---')
    
    for k in sorted(df_per_seed['k_shot'].unique()):
        # Get single model F1 per seed
        m1_f1 = df_per_seed[(df_per_seed['strategy'] == 'M1_single') &
                            (df_per_seed['k_shot'] == k) &
                            (df_per_seed['test_set'] == ts)]['f1'].values
        m4_f1 = df_per_seed[(df_per_seed['strategy'] == 'M2_single') &
                            (df_per_seed['k_shot'] == k) &
                            (df_per_seed['test_set'] == ts)]['f1'].values
        
        if len(m1_f1) == 0 or len(m4_f1) == 0:
            continue
        
        # Best single model per seed
        best_single = np.maximum(m1_f1, m4_f1)
        
        for strat in STRATEGIES:
            ens_f1 = df_per_seed[(df_per_seed['strategy'] == strat) &
                                 (df_per_seed['k_shot'] == k) &
                                 (df_per_seed['test_set'] == ts)]['f1'].values
            
            if len(ens_f1) < 5:
                continue
            
            diff = ens_f1 - best_single
            mean_diff = diff.mean()
            
            # Wilcoxon (only if there's variation)
            if np.all(diff == 0):
                p_val = 1.0
                sig = 'ns (identical)'
            else:
                try:
                    stat, p_val = wilcoxon(ens_f1, best_single, alternative='two-sided')
                    sig = '*' if p_val < 0.05 else 'ns'
                except ValueError:
                    p_val = 1.0
                    sig = 'ns (no var)'
            
            strat_label = STRATEGY_LABELS[strat]
            print(f'  k={k}  {strat_label:20s}  '
                  f'ΔF1={mean_diff:+.4f}  p={p_val:.4f} {sig}')

In [ ]:
# Test: Internal vs External per ensemble strategy (Mann-Whitney U)
print('\n' + '=' * 80)
print('STATISTICAL TESTS: Internal vs External per Strategy (Mann-Whitney U)')
print('=' * 80)

for strat in all_strats:
    strat_label = STRATEGY_LABELS.get(strat, strat)
    print(f'\n  {strat_label}:')
    
    for k in sorted(df_per_seed['k_shot'].unique()):
        int_f1 = df_per_seed[(df_per_seed['strategy'] == strat) &
                             (df_per_seed['k_shot'] == k) &
                             (df_per_seed['test_set'] == ts_int)]['f1'].values
        ext_f1 = df_per_seed[(df_per_seed['strategy'] == strat) &
                             (df_per_seed['k_shot'] == k) &
                             (df_per_seed['test_set'] == ts_ext)]['f1'].values
        
        if len(int_f1) < 3 or len(ext_f1) < 3:
            continue
        
        try:
            stat, p_val = mannwhitneyu(int_f1, ext_f1, alternative='two-sided')
            sig = '*' if p_val < 0.05 else 'ns'
        except ValueError:
            p_val = 1.0
            sig = 'ns'
        
        print(f'    k={k}  Int={int_f1.mean():.3f}  Ext={ext_f1.mean():.3f}  '
              f'p={p_val:.4f} {sig}')

## 18 — Net Improvement Analysis

In [ ]:
# Net improvement = F1_ensemble - max(F1_M1, F1_M4) per k
print('NET IMPROVEMENT: F1_ensemble - max(F1_M1, F1_M4)')
print('Positive = ensemble is better than best single model')
print('=' * 80)

for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'\n--- {short_ts} ---')
    
    for k in sorted(summary_df['k_shot'].unique()):
        m1 = summary_df[(summary_df['strategy'] == 'M1_single') &
                        (summary_df['k_shot'] == k) &
                        (summary_df['test_set'] == ts)]
        m4 = summary_df[(summary_df['strategy'] == 'M2_single') &
                        (summary_df['k_shot'] == k) &
                        (summary_df['test_set'] == ts)]
        
        if len(m1) == 0 or len(m4) == 0:
            continue
        
        best_single_f1 = max(m1.iloc[0]['f1_mean'], m4.iloc[0]['f1_mean'])
        best_single_name = 'M1' if m1.iloc[0]['f1_mean'] >= m4.iloc[0]['f1_mean'] else 'M4'
        
        print(f'  k={k}  best_single={best_single_name}(F1={best_single_f1:.3f})')
        
        for strat in STRATEGIES:
            ens = summary_df[(summary_df['strategy'] == strat) &
                            (summary_df['k_shot'] == k) &
                            (summary_df['test_set'] == ts)]
            if len(ens) > 0:
                delta = ens.iloc[0]['f1_mean'] - best_single_f1
                marker = '✓' if delta > 0 else '✗'
                print(f'    {STRATEGY_LABELS[strat]:20s}  '
                      f'F1={ens.iloc[0]["f1_mean"]:.3f}  '
                      f'Δ={delta:+.3f} {marker}')

## 19 — Hypothesis Testing Summary

In [ ]:
print('=' * 80)
print('HYPOTHESIS TESTING SUMMARY')
print('=' * 80)

ts_int = 'CheXpert Plus (Internal)'
ts_ext = 'RS UNAIR (External)'

# H1: Ensemble F1 > single model
print('\n--- H1: Dual-agent ensemble (M1+M4) improves F1 over single model ---')
for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'\n  {short_ts}:')
    
    for strat in STRATEGIES:
        strat_label = STRATEGY_LABELS[strat]
        ens_best = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == ts)]
        m1_best = best_df[(best_df['strategy'] == 'M1_single') & (best_df['test_set'] == ts)]
        m4_best = best_df[(best_df['strategy'] == 'M2_single') & (best_df['test_set'] == ts)]
        
        if len(ens_best) == 0 or len(m1_best) == 0 or len(m4_best) == 0:
            continue
        
        best_single = max(m1_best.iloc[0]['f1_mean'], m4_best.iloc[0]['f1_mean'])
        ens_f1 = ens_best.iloc[0]['f1_mean']
        delta = ens_f1 - best_single
        verdict = 'SUPPORTED' if delta > 0 else 'NOT SUPPORTED'
        
        print(f'    {strat_label:20s}  F1={ens_f1:.3f} vs best_single={best_single:.3f}  '
              f'Δ={delta:+.3f}  → {verdict}')

# H2: Ensemble has better PBR
print('\n--- H2: Ensemble achieves PBR closer to 1.0 ---')
for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'\n  {short_ts}:')
    
    for strat in STRATEGIES:
        strat_label = STRATEGY_LABELS[strat]
        ens = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == ts)]
        m1 = best_df[(best_df['strategy'] == 'M1_single') & (best_df['test_set'] == ts)]
        m4 = best_df[(best_df['strategy'] == 'M2_single') & (best_df['test_set'] == ts)]
        
        if len(ens) == 0 or len(m1) == 0 or len(m4) == 0:
            continue
        
        ens_pbr = ens.iloc[0]['pbr_mean']
        m1_pbr = m1.iloc[0]['pbr_mean']
        m4_pbr = m4.iloc[0]['pbr_mean']
        
        ens_dist = abs(ens_pbr - 1.0)
        best_single_dist = min(abs(m1_pbr - 1.0), abs(m4_pbr - 1.0))
        verdict = 'SUPPORTED' if ens_dist < best_single_dist else 'NOT SUPPORTED'
        
        print(f'    {strat_label:20s}  PBR={ens_pbr:.3f} (dist={ens_dist:.3f}) vs '
              f'best_single_dist={best_single_dist:.3f}  → {verdict}')

# H3: Cross-institutional robustness
print('\n--- H3: Ensemble reduces generalization gap ---')
for strat in STRATEGIES:
    strat_label = STRATEGY_LABELS[strat]
    ens_int = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == ts_int)]
    ens_ext = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == ts_ext)]
    m1_int = best_df[(best_df['strategy'] == 'M1_single') & (best_df['test_set'] == ts_int)]
    m1_ext = best_df[(best_df['strategy'] == 'M1_single') & (best_df['test_set'] == ts_ext)]
    
    if len(ens_int) == 0 or len(ens_ext) == 0 or len(m1_int) == 0 or len(m1_ext) == 0:
        continue
    
    ens_gap = abs(ens_int.iloc[0]['f1_mean'] - ens_ext.iloc[0]['f1_mean'])
    m1_gap = abs(m1_int.iloc[0]['f1_mean'] - m1_ext.iloc[0]['f1_mean'])
    verdict = 'SUPPORTED' if ens_gap < m1_gap else 'NOT SUPPORTED'
    
    print(f'  {strat_label:20s}  ens_gap={ens_gap:.3f} vs M1_gap={m1_gap:.3f}  → {verdict}')

## 20 — Comparison with M1+M2 Ensemble (if available)

In [ ]:
# Load M1+M2 results for comparison
m1m2_results_path = Path('results-ensemble/ensemble_summary_mean_std.csv')

if m1m2_results_path.exists():
    m1m2_summary = pd.read_csv(m1m2_results_path)
    
    print('COMPARISON: M1+M4 vs M1+M2 Ensemble')
    print('=' * 80)
    
    for ts in [ts_int, ts_ext]:
        short_ts = 'Internal' if 'Internal' in ts else 'External'
        print(f'\n--- {short_ts} ---')
        print(f'{"Strategy":20s}  {"M1+M2 F1":>12s}  {"M1+M4 F1":>12s}  {"ΔF1":>8s}  Winner')
        print('-' * 70)
        
        for strat in STRATEGIES:
            strat_label = STRATEGY_LABELS[strat]
            
            # Best F1 for M1+M2
            m1m2 = m1m2_summary[(m1m2_summary['strategy'] == strat) &
                                (m1m2_summary['test_set'] == ts)]
            if len(m1m2) > 0:
                m1m2_best_f1 = m1m2['f1_mean'].max()
            else:
                m1m2_best_f1 = None
            
            # Best F1 for M1+M4
            m1m4 = summary_df[(summary_df['strategy'] == strat) &
                              (summary_df['test_set'] == ts)]
            if len(m1m4) > 0:
                m1m4_best_f1 = m1m4['f1_mean'].max()
            else:
                m1m4_best_f1 = None
            
            if m1m2_best_f1 is not None and m1m4_best_f1 is not None:
                delta = m1m4_best_f1 - m1m2_best_f1
                winner = 'M1+M4' if delta > 0 else 'M1+M2' if delta < 0 else 'TIE'
                print(f'{strat_label:20s}  {m1m2_best_f1:12.3f}  {m1m4_best_f1:12.3f}  '
                      f'{delta:+8.3f}  {winner}')
            else:
                print(f'{strat_label:20s}  {"N/A":>12s}  {"N/A":>12s}')
else:
    print('M1+M2 ensemble results not found at:', m1m2_results_path)
    print('Run 5-ensemble-p1.ipynb first to generate comparison data.')

## 21 — Export All Results

In [ ]:
# Summary of exported files
print('EXPORTED CSV FILES:')
for f in sorted(RESULTS_DIR.glob('*.csv')):
    print(f'  {f.name}')

print(f'\nEXPORTED FIGURES:')
for f in sorted(FIGURES_DIR.glob('fig*')):
    print(f'  {f.name}')

## 22 — Key Findings Summary

In [ ]:
print('=' * 80)
print('KEY FINDINGS — Dual-Agent Ensemble M1+M4 (P1)')
print('=' * 80)

# Overall best
for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    sub = summary_df[summary_df['test_set'] == ts]
    
    # Best overall strategy
    ens_sub = sub[~sub['strategy'].str.contains('single')]
    best_idx = ens_sub['f1_mean'].idxmax()
    best = ens_sub.loc[best_idx]
    
    # Best single model
    single_sub = sub[sub['strategy'].str.contains('single')]
    best_single_idx = single_sub['f1_mean'].idxmax()
    best_single = single_sub.loc[best_single_idx]
    best_single_name = 'M1' if best_single['strategy'] == 'M1_single' else 'M4'
    
    delta = best['f1_mean'] - best_single['f1_mean']
    
    print(f'\n{short_ts}:')
    print(f'  Best ensemble:  {STRATEGY_LABELS[best["strategy"]]} at k={int(best["k_shot"])}  '
          f'F1={best["f1_mean"]:.3f}±{best["f1_std"]:.3f}')
    print(f'  Best single:    {best_single_name} at k={int(best_single["k_shot"])}  '
          f'F1={best_single["f1_mean"]:.3f}±{best_single["f1_std"]:.3f}')
    print(f'  Net improvement: {delta:+.3f} {"✓" if delta > 0 else "✗"}')

print('\n' + '-' * 80)
print('INTERPRETASI:')
print('-' * 80)
print('1. M4 sebagai partner ensemble JAUH lebih baik dari M2:')
print('   - M4 memiliki sensitivity yang meaningful (vs M2 yang "dead agent")')
print('   - Kedua model aktif berkontribusi → ensemble benar-benar aggregasi 2 agen')
print()
print('2. Strategi S2 (Any Positive) dan S4 (PBR-Weighted) diharapkan menarik:')
print('   - S2 mengkapitalisasi sensitivity kedua model → coverage PTX meningkat')
print('   - S4 balance antara M1 (over-predict) dan M4 (under-predict)')
print()
print('3. Cross-architectural ensemble (Qwen2.5-VL + Gemma 3):')
print('   - Diversitas arsitektur bisa mengurangi correlated errors')
print('   - M4 yang robust cross-institutional bisa menstabilkan ensemble')